[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/06_End_to_End_Project_Return_Prediction.ipynb)

# Module 6: End-to-End Project - E-commerce Order Return Prediction (30 min)

Returns cost Indian e-commerce companies a lot in reverse logistics, repackaging and lost sales, especially in fashion. In this project you build a complete solution on a dataset of 10,000 orders, using everything from Modules 1 to 5.

## 6.1 Business problem and dataset

**Goal**: score every order right after delivery (when the delivery time and the customer's rating are known) with a return-risk probability, so the support team can call high-risk customers and offer an exchange, size help or a fix before a return is raised.

**Dataset**: `ecommerce_orders_10k.xlsx`, sheet `Orders`, 10,000 orders from calendar year 2025 across 20 Indian cities and 7 categories. The sheet `Data_Dictionary` explains every column. The data is realistic but simulated, and it deliberately contains missing values for cleaning practice.

| Column | Description |
|---|---|
| Order_ID, Order_Date, Customer_ID | Order identifier, date-time of order, customer identifier |
| Customer_Age, Gender, City, City_Tier | Customer details (Age has some missing values) |
| Category, Product_Name | One of 7 categories and 41 products |
| Unit_Price, Quantity, Discount_Pct, Order_Value | Pricing (Order_Value = price x quantity x (1 - discount)) |
| Payment_Method, Device_Type, Is_Member | UPI, cards, COD, net banking, wallet; app, mobile web or desktop; membership |
| Previous_Orders, Session_Duration_Min | Customer history and browsing time |
| Delivery_Days, Customer_Rating, Review_Text | Post-delivery information (rating and review have missing values) |
| Returned | TARGET: Yes / No (about 18% Yes) |


![Figure 13: Project architecture.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d13_project.png)
*Figure 13: Project architecture.*


> **How to run the project in Colab**
>
> 1. Open a new Colab notebook. 2. Run Step 0; a file picker appears; choose `ecommerce_orders_10k.xlsx` (alternatively drag the file into the Files panel on the left before running). 3. Run Steps 1 to 14 in order, each in its own cell. Later steps use variables created by earlier steps. Total run time is about 2 to 3 minutes on the free CPU runtime.

## Step 0: Setup and data upload

In [ ]:
# STEP 0: Imports and data loading
import os
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

FILE = "ecommerce_orders_10k.xlsx"
DATA_URL = "https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/data/ecommerce_orders_10k.xlsx"

# 1) use a local copy if it exists (repo cloned locally or file uploaded to Colab)
for candidate in [FILE, os.path.join("..", "data", FILE), os.path.join("data", FILE)]:
    if os.path.exists(candidate):
        FILE = candidate
        break
else:
    # 2) otherwise download it from the GitHub repo
    try:
        urllib.request.urlretrieve(DATA_URL, FILE)
        print("Downloaded dataset from GitHub")
    except Exception as err:
        # 3) last resort: upload it manually in Colab
        print("Download failed:", err)
        from google.colab import files
        files.upload()
print("Using file:", FILE, "| exists:", os.path.exists(FILE))

## Step 1: Load and understand the data

Check the shape, data types, missing values and the balance of the target. About 18% of orders are returned, so the classes are imbalanced: a model that always says "No" would be 82% accurate and useless. This is why we will focus on recall, precision, F1 and ROC-AUC rather than accuracy.

In [ ]:
# STEP 1: Load and understand the data
df = pd.read_excel(FILE, sheet_name="Orders")
print("Shape:", df.shape)
print(df.head())
print()
df.info()
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nTarget distribution:\n", df["Returned"].value_counts(normalize=True).round(3))
print("\nNumeric summary:\n", df.describe(include="number").round(1).T)

## Step 2: Clean the data

Missing ages are filled with the median, missing devices become "Unknown", and missing ratings are filled with the median rating of the same category (a smarter rule than the global median). The target is converted to 0/1.

In [ ]:
# STEP 2: Clean the data
df = df.drop_duplicates(subset="Order_ID").copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

# Fill missing values with sensible, explainable rules
df["Customer_Age"] = df["Customer_Age"].fillna(df["Customer_Age"].median())
df["Device_Type"] = df["Device_Type"].fillna("Unknown")
df["Customer_Rating"] = df["Customer_Rating"].fillna(
    df.groupby("Category")["Customer_Rating"].transform("median"))   # median of same category
df["Review_Text"] = df["Review_Text"].fillna("No review")

# Target to 0/1
df["Returned_Flag"] = (df["Returned"] == "Yes").astype(int)

# Sanity checks
assert df["Order_Value"].min() > 0
assert df[["Customer_Age", "Device_Type", "Customer_Rating"]].isna().sum().sum() == 0
print("Rows after cleaning:", len(df))
print("Remaining missing values:", df.isna().sum().sum())
print(f"Overall return rate: {df['Returned_Flag'].mean():.1%}")

## Step 3: Exploratory data analysis

Charts reveal the patterns the model will learn: Fashion has by far the highest return rate, COD orders return more, returns rise with delivery days, and low ratings are a strong signal.

In [ ]:
# STEP 3: Exploratory data analysis (EDA)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

rate = df.groupby("Category")["Returned_Flag"].mean().sort_values(ascending=False) * 100
rate.plot(kind="bar", ax=axes[0, 0], color="indianred")
axes[0, 0].set_title("Return rate by category (%)"); axes[0, 0].tick_params(axis="x", rotation=45)

pay = df.groupby("Payment_Method")["Returned_Flag"].mean().sort_values(ascending=False) * 100
pay.plot(kind="bar", ax=axes[0, 1], color="darkorange")
axes[0, 1].set_title("Return rate by payment method (%)"); axes[0, 1].tick_params(axis="x", rotation=45)

dd = df.groupby("Delivery_Days")["Returned_Flag"].mean() * 100
dd.plot(marker="o", ax=axes[0, 2], color="purple")
axes[0, 2].set_title("Return rate vs delivery days (%)")

rr = df.groupby("Customer_Rating")["Returned_Flag"].mean() * 100
rr.plot(kind="bar", ax=axes[1, 0], color="teal")
axes[1, 0].set_title("Return rate by customer rating (%)")

sns.histplot(np.log10(df["Order_Value"]), bins=50, ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Order value (log10 scale)")

num_cols = ["Customer_Age", "Unit_Price", "Quantity", "Discount_Pct", "Order_Value",
            "Previous_Orders", "Session_Duration_Min", "Delivery_Days", "Customer_Rating",
            "Returned_Flag"]
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0, ax=axes[1, 2], annot=False)
axes[1, 2].set_title("Correlation heatmap")
plt.tight_layout()
plt.show()

# Monthly revenue trend
monthly = df.set_index("Order_Date").resample("MS")["Order_Value"].sum() / 1e5
monthly.plot(marker="o", figsize=(12, 4), title="Monthly revenue (Rs lakh)")
plt.show()
city_revenue = df.groupby("City")["Order_Value"].sum() / 1e5
print("Top 5 cities by revenue (Rs lakh):\n", city_revenue.nlargest(5).round(1))

## Step 4: Statistical tests

Before trusting the patterns we test them (Module 2). Chi-square tests confirm that category and payment method are related to returns, a Welch t-test confirms that returned orders had longer deliveries, and conditional probabilities quantify the risk.

In [ ]:
# STEP 4: Statistical tests - are the patterns real or just noise?
# 1) Chi-square: is Category related to Returned?
ct = pd.crosstab(df["Category"], df["Returned_Flag"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Category vs Returned       : chi2={chi2:.1f}, p={p:.2e}")

# 2) Chi-square: is Payment_Method related to Returned?
ct = pd.crosstab(df["Payment_Method"], df["Returned_Flag"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Payment method vs Returned : chi2={chi2:.1f}, p={p:.2e}")

# 3) Welch t-test: do returned orders have longer delivery times?
ret = df.loc[df["Returned_Flag"] == 1, "Delivery_Days"]
kept = df.loc[df["Returned_Flag"] == 0, "Delivery_Days"]
t, p = stats.ttest_ind(ret, kept, equal_var=False)
print(f"Delivery days (returned {ret.mean():.2f} vs kept {kept.mean():.2f}): t={t:.2f}, p={p:.2e}")

# 4) Bayes-style conditional probabilities
p_ret = df["Returned_Flag"].mean()
p_ret_cod = df.loc[df["Payment_Method"] == "COD", "Returned_Flag"].mean()
fashion_low = (df["Category"] == "Fashion") & (df["Customer_Rating"] <= 2)
p_ret_fashion_low = df.loc[fashion_low, "Returned_Flag"].mean()
print(f"\nP(return)                         = {p_ret:.3f}")
print(f"P(return | COD)                   = {p_ret_cod:.3f}")
print(f"P(return | Fashion & rating <= 2) = {p_ret_fashion_low:.3f}")

# 5) 95% confidence interval for the overall return rate
n = len(df)
se = np.sqrt(p_ret * (1 - p_ret) / n)
print(f"95% CI for return rate: {p_ret - 1.96 * se:.3f} to {p_ret + 1.96 * se:.3f}")

## Step 5: Feature engineering

New features capture business knowledge: time features (month, weekday, hour, weekend, sale month), flags for high discount, slow delivery, new customer and COD, and a log transform of the skewed order value. Identifier columns and free text are not used as model inputs.

In [ ]:
# STEP 5: Feature engineering
# Business use case: score every order right after delivery + rating, so the support
# team can reach out (exchange, size help, fix) BEFORE the customer raises a return.
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Weekday"] = df["Order_Date"].dt.dayofweek
df["Order_Hour"] = df["Order_Date"].dt.hour
df["Is_Weekend"] = (df["Order_Weekday"] >= 5).astype(int)
df["Is_Sale_Month"] = df["Order_Month"].isin([1, 8, 10, 11]).astype(int)
df["High_Discount"] = (df["Discount_Pct"] > 40).astype(int)
df["Slow_Delivery"] = (df["Delivery_Days"] > 5).astype(int)
df["Is_New_Customer"] = (df["Previous_Orders"] == 0).astype(int)
df["Log_Order_Value"] = np.log1p(df["Order_Value"])
df["Is_COD"] = (df["Payment_Method"] == "COD").astype(int)

numeric_features = ["Customer_Age", "City_Tier", "Unit_Price", "Quantity", "Discount_Pct",
                    "Log_Order_Value", "Previous_Orders", "Session_Duration_Min", "Delivery_Days",
                    "Customer_Rating", "Order_Month", "Order_Weekday", "Order_Hour", "Is_Weekend",
                    "Is_Sale_Month", "High_Discount", "Slow_Delivery", "Is_New_Customer", "Is_COD"]
categorical_features = ["Gender", "Category", "Payment_Method", "Device_Type", "Is_Member"]
TARGET = "Returned_Flag"

X = df[numeric_features + categorical_features]
y = df[TARGET]
print("Feature matrix:", X.shape, "| positives:", y.sum())

## Step 6: Split and preprocess

A stratified 80/20 split keeps the 18% return rate in both sets. The `ColumnTransformer` scales numeric features and one-hot encodes categorical features, learning only from the training data.

In [ ]:
# STEP 6: Train/test split and preprocessing pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)
print(f"Return rate train {y_train.mean():.3f} | test {y_test.mean():.3f}")

## Step 7: Machine learning models

We train logistic regression, Random Forest and gradient boosting. `class_weight="balanced"` tells each model to pay more attention to the minority class (returns).

In [ ]:
# STEP 7: Train and compare machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced_subsample",
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                        class_weight="balanced", random_state=42),
}

def score_model(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"model": name,
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "roc_auc": roc_auc_score(y_true, proba)}

results, probas = [], {}
for name, model in models.items():
    model.fit(X_train_p, y_train)
    probas[name] = model.predict_proba(X_test_p)[:, 1]
    results.append(score_model(name, y_test, probas[name]))

results_df = pd.DataFrame(results).set_index("model").round(3)
print(results_df)

best_ml = results_df["roc_auc"].idxmax()
print(f"\nBest ML model by ROC-AUC: {best_ml}")
best_pred = (probas[best_ml] >= 0.5).astype(int)
print(classification_report(y_test, best_pred, target_names=["Kept", "Returned"]))

## Step 8: Business-driven threshold

A probability of 0.5 is not automatically the right cut-off. We assume a call costs Rs 30 and a prevented return saves Rs 250, with 40% of contacted risky customers saved. The table shows the trade-off between catching more returns and making more calls, and we pick the threshold with the highest net saving.

In [ ]:
# STEP 8: Choose a decision threshold that suits the business
# Calling a customer costs ~Rs 30; a prevented return saves ~Rs 250 of reverse logistics.
from sklearn.metrics import confusion_matrix

proba = probas[best_ml]
rows = []
for th in np.arange(0.30, 0.81, 0.05):
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= th).astype(int)).ravel()
    saving = tp * 250 * 0.4 - (tp + fp) * 30      # assume 40% of contacted risky orders are saved
    rows.append({"threshold": round(th, 2), "flagged": tp + fp, "caught_returns": tp,
                 "missed_returns": fn, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn),
                 "net_saving_Rs": saving})
th_df = pd.DataFrame(rows).round(3)
print(th_df.to_string(index=False))
best_th = th_df.loc[th_df["net_saving_Rs"].idxmax(), "threshold"]
print(f"\nBest threshold for net saving: {best_th}")

## Step 9: Deep learning model

A Keras network with two hidden layers, batch normalisation and dropout is trained on the same preprocessed features, with class weights and early stopping on validation AUC.

In [ ]:
# STEP 9: Deep learning model (Keras neural network)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

n_features = X_train_p.shape[1]
nn = keras.Sequential([
    keras.Input(shape=(n_features,)),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
           loss="binary_crossentropy",
           metrics=[keras.metrics.AUC(name="auc")])

# Returns are the minority class -> give them more weight
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}

early_stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                           restore_best_weights=True)
history = nn.fit(X_train_p, y_train.values, validation_split=0.2, epochs=60, batch_size=64,
                 class_weight=class_weight, callbacks=[early_stop], verbose=0)
print("Epochs trained:", len(history.history["loss"]))

probas["Neural Network"] = nn.predict(X_test_p, verbose=0).ravel()
nn_scores = score_model("Neural Network", y_test, probas["Neural Network"])
nn_scores.pop("model")
results_df.loc["Neural Network"] = pd.Series(nn_scores).round(3)
print(results_df)

plt.figure(figsize=(8, 4))
plt.plot(history.history["auc"], label="train AUC")
plt.plot(history.history["val_auc"], label="validation AUC")
plt.xlabel("Epoch"); plt.title("Neural network training"); plt.legend()
plt.show()

## Step 10: Compare all models

In [ ]:
# STEP 10: Compare all models with ROC curves and confusion matrices
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, pr in probas.items():
    fpr, tpr, _ = roc_curve(y_test, pr)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, pr):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves"); axes[0].legend()

ConfusionMatrixDisplay.from_predictions(y_test, (probas[best_ml] >= 0.5).astype(int),
                                        display_labels=["Kept", "Returned"], cmap="Blues", ax=axes[1])
axes[1].set_title(f"{best_ml} - confusion matrix")
plt.tight_layout()
plt.show()
print(results_df.sort_values("roc_auc", ascending=False))

> **Interpretation**
>
> All models reach a ROC-AUC of about 0.72 to 0.77. Logistic regression is the best here and catches about 70% of returns, with the other models close behind. On structured business tables with around 10,000 rows, simple models and gradient-boosted trees often match or beat neural networks; deep learning shines on images, audio and text. Exact numbers can vary slightly between runs and library versions.

## Step 11: Feature importance and business insights

In [ ]:
# STEP 11: What drives returns? Feature importance + business insights
rf = models["Random Forest"]
importance = (pd.Series(rf.feature_importances_, index=feature_names)
                .sort_values(ascending=False).head(15))
importance.index = importance.index.str.replace("num__", "").str.replace("cat__", "")
importance.sort_values().plot(kind="barh", figsize=(9, 6), color="seagreen",
                              title="Top 15 drivers of returns (Random Forest)")
plt.show()

print("BUSINESS INSIGHTS")
fashion = df.loc[df.Category == "Fashion", "Returned_Flag"].mean()
grocery = df.loc[df.Category == "Grocery", "Returned_Flag"].mean()
print(f"1. Fashion return rate {fashion:.0%} vs Grocery {grocery:.0%} -> add size guides / fit tools.")
print(f"2. COD orders return {p_ret_cod:.0%} vs overall {p_ret:.0%} -> nudge UPI with prepaid offers.")
slow = df.loc[df.Slow_Delivery == 1, 'Returned_Flag'].mean()
fast = df.loc[df.Slow_Delivery == 0, 'Returned_Flag'].mean()
print(f"3. Slow deliveries (>5 days) return {slow:.0%} vs {fast:.0%} -> prioritise tier-3 logistics.")
low = df.loc[df.Customer_Rating <= 2, 'Returned_Flag'].mean()
print(f"4. Ratings of 1-2 stars return {low:.0%} -> trigger instant support call on low ratings.")

## Step 12: Save the model and score new orders

The full pipeline (preprocessing + model) is saved with `joblib` so it can be loaded in an API or a batch job. A helper function rebuilds the engineered features for new orders, and each order gets a risk score and an action.

In [ ]:
# STEP 12: Save the best pipeline and score new orders
import joblib
from sklearn.pipeline import Pipeline

final_model = Pipeline([("prep", preprocessor), ("model", models[best_ml])])
final_model.fit(X_train, y_train)                 # refit the whole pipeline on raw features
joblib.dump(final_model, "return_risk_model.joblib")
print("Model saved: return_risk_model.joblib")

def add_features(orders):
    o = orders.copy()
    o["Order_Date"] = pd.to_datetime(o["Order_Date"])
    o["Order_Month"] = o["Order_Date"].dt.month
    o["Order_Weekday"] = o["Order_Date"].dt.dayofweek
    o["Order_Hour"] = o["Order_Date"].dt.hour
    o["Is_Weekend"] = (o["Order_Weekday"] >= 5).astype(int)
    o["Is_Sale_Month"] = o["Order_Month"].isin([1, 8, 10, 11]).astype(int)
    o["High_Discount"] = (o["Discount_Pct"] > 40).astype(int)
    o["Slow_Delivery"] = (o["Delivery_Days"] > 5).astype(int)
    o["Is_New_Customer"] = (o["Previous_Orders"] == 0).astype(int)
    o["Order_Value"] = o["Unit_Price"] * o["Quantity"] * (1 - o["Discount_Pct"] / 100)
    o["Log_Order_Value"] = np.log1p(o["Order_Value"])
    o["Is_COD"] = (o["Payment_Method"] == "COD").astype(int)
    return o[numeric_features + categorical_features]

new_orders = pd.DataFrame([
    {"Order_Date": "2026-01-10 21:15", "Customer_Age": 22, "Gender": "Female", "City_Tier": 3,
     "Category": "Fashion", "Unit_Price": 1499, "Quantity": 2, "Discount_Pct": 45,
     "Payment_Method": "COD", "Device_Type": "Mobile App", "Is_Member": "No", "Previous_Orders": 0,
     "Session_Duration_Min": 6.5, "Delivery_Days": 8, "Customer_Rating": 2},
    {"Order_Date": "2026-01-12 10:05", "Customer_Age": 38, "Gender": "Male", "City_Tier": 1,
     "Category": "Books", "Unit_Price": 549, "Quantity": 1, "Discount_Pct": 10,
     "Payment_Method": "UPI", "Device_Type": "Desktop", "Is_Member": "Yes", "Previous_Orders": 14,
     "Session_Duration_Min": 18.0, "Delivery_Days": 2, "Customer_Rating": 5},
])
loaded = joblib.load("return_risk_model.joblib")
risk = loaded.predict_proba(add_features(new_orders))[:, 1]
for (_, o), r in zip(new_orders.iterrows(), risk):
    action = "CALL CUSTOMER" if r >= best_th else "no action"
    print(f"{o['Category']:8} | {o['Payment_Method']:4} | rating {o['Customer_Rating']} "
          f"-> return risk {r:.0%} -> {action}")

## Step 13 (bonus): Customer segmentation

Customers are aggregated to Recency, Frequency and Monetary values and clustered with K-Means (Module 4). The log transform reduces the effect of a few very large customers.

In [ ]:
# STEP 13 (bonus): Customer segmentation with K-Means (RFM)
from sklearn.cluster import KMeans

snapshot = df["Order_Date"].max() + pd.Timedelta(days=1)
rfm = df.groupby("Customer_ID").agg(
    Recency=("Order_Date", lambda d: (snapshot - d.max()).days),
    Frequency=("Order_ID", "count"),
    Monetary=("Order_Value", "sum"),
    Return_Rate=("Returned_Flag", "mean"))

rfm_scaled = StandardScaler().fit_transform(np.log1p(rfm[["Recency", "Frequency", "Monetary"]]))
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm["Segment"] = kmeans.fit_predict(rfm_scaled)

profile = rfm.groupby("Segment").agg(Customers=("Recency", "size"), Recency=("Recency", "mean"),
                                     Frequency=("Frequency", "mean"), Monetary=("Monetary", "mean"),
                                     Return_Rate=("Return_Rate", "mean")).round(2)
profile = profile.sort_values("Monetary", ascending=False)
profile["Label"] = ["Champions", "Loyal", "Potential", "At risk / one-time"]
print(profile)

## Step 14 (GenAI bridge): semantic search over reviews

The final step connects the project to Generative AI. Review texts are converted to TF-IDF vectors and searched with cosine similarity (Module 3), which is the retrieval half of a RAG system. The retrieved reviews are then placed into a prompt that can be sent to any LLM to explain why customers return products. In the GenAI program, TF-IDF will be replaced by neural embeddings and a vector database.

In [ ]:
# STEP 14 (GenAI bridge): semantic search over customer reviews
# This is the "retrieval" half of a RAG chatbot that answers: "Why are customers returning X?"
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cols = ["Order_ID", "Category", "Review_Text", "Returned"]
reviews = df.loc[df["Review_Text"] != "No review", cols].reset_index(drop=True)
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
review_vectors = vectorizer.fit_transform(reviews["Review_Text"])
print("Review vectors:", review_vectors.shape)

def find_reviews(query, top_k=5, category=None):
    scores = cosine_similarity(vectorizer.transform([query]), review_vectors).ravel()
    result = reviews.assign(score=scores)
    if category:
        result = result[result["Category"] == category]
    result = result.sort_values("score", ascending=False).drop_duplicates("Review_Text")
    return result.head(top_k)

print("\nQuery: 'size did not fit' in Fashion")
show = ["Order_ID", "Review_Text", "score"]
print(find_reviews("size did not fit", category="Fashion")[show].to_string(index=False))

print("\nQuery: 'damaged product not working' in Electronics")
print(find_reviews("damaged product not working", category="Electronics")[show].to_string(index=False))

# Build an LLM-ready prompt from returned-order reviews (send it to any LLM API)
sample = (reviews[(reviews["Category"] == "Fashion") & (reviews["Returned"] == "Yes")]["Review_Text"]
          .drop_duplicates().sample(10, random_state=1))
context = "\n".join("- " + t for t in sample)
prompt = ("You are an e-commerce analyst. Using only the reviews below, list the top reasons "
          f"customers return Fashion products and suggest 3 fixes.\n\nReviews:\n{context}")
print("\nPrompt for the LLM:\n", prompt)

## 6.2 Project summary and extensions

| Module | Where it was used in the project |
|---|---|
| Python | Pandas cleaning, functions for feature engineering, dictionaries of models, joblib, f-string prompts |
| Statistics | Missing-value strategy, chi-square and t-tests, conditional probabilities, confidence interval |
| Linear Algebra | Feature matrix, scaling, one-hot vectors, cosine similarity for review search |
| Machine Learning | Pipeline, logistic regression, Random Forest, gradient boosting, metrics, threshold, K-Means |
| Deep Learning | Keras network with dropout, batch normalisation, class weights and early stopping |
| GenAI | Retrieval over reviews and an LLM-ready prompt |


**Try these extensions**: add `Product_Name` as a feature and check whether AUC improves; tune the Random Forest with `GridSearchCV`; replace TF-IDF with a sentence-embedding model (`pip install sentence-transformers`) and compare search results; send the Step 14 prompt to an LLM API and review its answer.